In [2]:
%pip install tqdm pyarrow pandas


Note: you may need to restart the kernel to use updated packages.


In [8]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm


In [11]:
DATA_DIR = Path("../data/raw/ctssb/ctssb_data_1M")
files = sorted(DATA_DIR.glob("file-*.jsonl.gz"))
print(len(files)) 
files[:3]


10


[WindowsPath('../data/raw/ctssb/ctssb_data_1M/file-0.jsonl.gz'),
 WindowsPath('../data/raw/ctssb/ctssb_data_1M/file-1.jsonl.gz'),
 WindowsPath('../data/raw/ctssb/ctssb_data_1M/file-2.jsonl.gz')]

In [10]:
import os; print(os.getcwd())

C:\Users\Jahs\Desktop\CodeGuard\notebooks


In [12]:
sample = pd.read_json(files[0], lines=True, compression="gzip")
print(sample.shape)
print(sample.columns.tolist())
print(sample.iloc[0].to_dict()) 


(100000, 13)
['project', 'commit_sha', 'parent_sha', 'file_path', 'project_url', 'likely_bug', 'comodified', 'in_function', 'diff', 'before', 'after', 'sstub_pattern', 'edit_script']
{'project': 'trac', 'commit_sha': 'f8b54ac7b9f5a51e8000bf5c51ee7ad44e342f67', 'parent_sha': 'ea9ee1a81bfa2495a3d94567a19896c9f694ddf2', 'file_path': 'trac/util/tests/datefmt.py', 'project_url': 'https://github.com/arielnetworks/trac', 'likely_bug': True, 'comodified': False, 'in_function': True, 'diff': '@@ -45,7 +45,7 @@ else:\n class DateFormatTestCase(unittest.TestCase):\n \n     def test_to_datetime(self):\n-        expected = datetime.datetime(1970,1,1,1,0,23,0,datefmt.localtz)\n+        expected = datetime.datetime.fromtimestamp(23, datefmt.localtz)\n         self.assertEqual(datefmt.to_datetime(23), expected)\n         self.assertEqual(datefmt.to_datetime(23L), expected)\n         self.assertEqual(datefmt.to_datetime(23.0), expected)\n', 'before': 'expected = datetime . datetime ( 1970 , 1 , 1 , 1 ,

In [13]:
columns = ["project", "file_path", "likely_bug", "in_function",
           "comodified", "diff", "before", "after", "sstub_pattern"]
parts = []
for path in tqdm(files):
    chunk = pd.read_json(path, lines=True, compression="gzip")
    parts.append(chunk[columns])   
    del chunk

df = pd.concat(parts, ignore_index=True)
del parts

print(df.shape) 
df.head(2)

  0%|          | 0/10 [00:00<?, ?it/s]

(966985, 9)


,project,file_path,likely_bug,in_function,comodified,diff,before,after,sstub_pattern
0,trac,trac/util/tests/datefmt.py,True,True,False,"@@ -45,7 +45,7 @@ else:\n class DateFormatTest...","expected = datetime . datetime ( 1970 , 1 , 1 ...",expected = datetime . datetime . fromtimestamp...,SINGLE_STMT
1,trac,trac/util/translation.py,True,True,False,"@@ -27,7 +27,7 @@ def gettext_noop(string, **k...","def ngettext_noop ( singular , plural , num ) ...","def ngettext_noop ( singular , plural , num , ...",SINGLE_STMT


In [14]:
print("rows:", len(df))
print(df["likely_bug"].value_counts())
print(df["in_function"].value_counts())
print(df["comodified"].value_counts())
print(df["sstub_pattern"].value_counts())
print("projects:", df["project"].nunique())

rows: 966985
likely_bug
True    966985
Name: count, dtype: int64
in_function
True     714241
False    252744
Name: count, dtype: int64
comodified
False    966985
Name: count, dtype: int64
sstub_pattern
SINGLE_STMT                       333076
CHANGE_STRING_LITERAL             194106
CHANGE_IDENTIFIER_USED             68884
CHANGE_BINARY_OPERAND              47638
SAME_FUNCTION_MORE_ARGS            41741
WRONG_FUNCTION_NAME                38582
CHANGE_NUMERIC_LITERAL             33265
ADD_FUNCTION_AROUND_EXPRESSION     31927
CHANGE_ATTRIBUTE_USED              30323
SINGLE_TOKEN                       25847
ADD_METHOD_CALL                    17424
MORE_SPECIFIC_IF                   15622
ADD_ELEMENTS_TO_ITERABLE           15014
SAME_FUNCTION_LESS_ARGS            14343
CHANGE_BOOLEAN_LITERAL             12922
ADD_ATTRIBUTE_ACCESS                9786
CHANGE_BINARY_OPERATOR              9193
SAME_FUNCTION_WRONG_CALLER          7718
CHANGE_KEYWORD_ARGUMENT_USED        6294
LESS_SPECIFIC_IF   

In [15]:
filtered = df[(df["likely_bug"] == True) & (df["in_function"] == True)].copy()
print(len(filtered))


714241


In [16]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
filtered.to_parquet("../data/processed/ctssb_filtered.parquet", index=False)


In [22]:

filtered = pd.read_parquet("../data/processed/ctssb_filtered.parquet")
print(len(filtered))


714241


In [23]:
def split_diff(diff: str):
    buggy, fixed = [], []
    for line in diff.rstrip("\n").split("\n"):
        if line.startswith("@@"): #header of block
            continue
        if line.startswith("\\"): # "\ No newline at end of file"
            continue
        if line.startswith("-"):  # only in version that contains bug
            buggy.append(line[1:])
        elif line.startswith("+"): # only in version that was fixed
            fixed.append(line[1:])
        else:
            buggy.append(line[1:])
            fixed.append(line[1:])
    return "\n".join(buggy), "\n".join(fixed)


In [24]:
row = filtered[filtered["diff"].str.contains("fromtimestamp(23, datefmt.localtz)", regex=False)].iloc[0]
b, f = split_diff(row["diff"])
print("datetime.datetime(1970,1,1,1,0,23,0,datefmt.localtz)" in b)
print("datetime.datetime.fromtimestamp(23, datefmt.localtz)" in f)
print(b); print("-----"); print(f)


True
True
class DateFormatTestCase(unittest.TestCase):

    def test_to_datetime(self):
        expected = datetime.datetime(1970,1,1,1,0,23,0,datefmt.localtz)
        self.assertEqual(datefmt.to_datetime(23), expected)
        self.assertEqual(datefmt.to_datetime(23L), expected)
        self.assertEqual(datefmt.to_datetime(23.0), expected)
-----
class DateFormatTestCase(unittest.TestCase):

    def test_to_datetime(self):
        expected = datetime.datetime.fromtimestamp(23, datefmt.localtz)
        self.assertEqual(datefmt.to_datetime(23), expected)
        self.assertEqual(datefmt.to_datetime(23L), expected)
        self.assertEqual(datefmt.to_datetime(23.0), expected)


In [26]:
pairs = filtered["diff"].apply(split_diff)
filtered["buggy"] = pairs.str[0]
filtered["fixed"] = pairs.str[1]
filtered[["buggy", "fixed"]].head()


,buggy,fixed
0,class DateFormatTestCase(unittest.TestCase):\n...,class DateFormatTestCase(unittest.TestCase):\n...
1,N_ = gettext_noop\ngettext = _ = gettext_noop\...,N_ = gettext_noop\ngettext = _ = gettext_noop\...
2,if 'exception_level_filters' in kwargs...,if 'exception_level_filters' in kwargs...
3,}\n\n if request.get_json():\n r...,}\n\n if request.get_json():\n r...
4,"scrubbed_qs_params = _scrub_obj(qs_params,...","scrubbed_qs_params = _scrub_obj(qs_params,..."


In [28]:
filtered.to_parquet("../data/processed/ctssb_snippets.parquet", index=False)
